<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

نکات مهم کد:
- مقدار threshold=0.30 صرفاً برای مثال است و باید با داده واقعی ارزیابی و تنظیم شود.
- این روش جمله‌ها را حذف یا نگه می‌دارد؛ متن جدیدی تولید نمی‌کند.
- در روش Abstractive Compression می‌توان از Local LLM برای خلاصه‌سازی اطلاعات مرتبط استفاده کرد.
- حذف جمله‌های مرتبط به دلیل Threshold نامناسب می‌تواند باعث از دست رفتن اطلاعات مهم شود.

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Knowledge Base
documents = [
    {
        "id": "HR-001",
        "text": """
        Employees receive 26 annual leave days per year.
        Leave requests must be approved by the manager.
        Annual leave cannot be transferred without HR approval.
        """,
    },
    {
        "id": "HR-002",
        "text": """
        Employees have supplementary health insurance.
        The insurance provider is selected by the HR department.
        Insurance coverage includes dental services.
        """,
    },
    {
        "id": "HR-003",
        "text": """
        Monthly salaries are paid at the end of each month.
        Salary adjustments are reviewed by the finance department.
        Employees can contact payroll for salary questions.
        """,
    },
    {
        "id": "HR-004",
        "text": """
        Employees can work remotely two days per week.
        Remote work requires manager approval.
        Employees must attend important office meetings.
        """,
    },
]

# 2. Initialize Embedding Model
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)


# 3. Initial Retrieval
def retrieve(query, top_k=4):

    document_texts = [doc["text"] for doc in documents]

    document_embeddings = model.encode(
        document_texts, convert_to_numpy=True, normalize_embeddings=True
    )

    query_embedding = model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )

    scores = cosine_similarity(query_embedding, document_embeddings)[0]

    indices = np.argsort(scores)[::-1][:top_k]

    return [
        {**documents[index], "retrieval_score": float(scores[index])}
        for index in indices
    ]


# 4. Context Compression
def compress_context(query, retrieved_documents, threshold=0.30):

    compressed_documents = []

    for document in retrieved_documents:
        sentences = [
            sentence.strip()
            for sentence in document["text"].split(".")
            if sentence.strip()
        ]

        sentence_embeddings = model.encode(
            sentences, convert_to_numpy=True, normalize_embeddings=True
        )

        query_embedding = model.encode(
            [query], convert_to_numpy=True, normalize_embeddings=True
        )

        scores = cosine_similarity(query_embedding, sentence_embeddings)[0]

        relevant_sentences = [
            sentence for sentence, score in zip(sentences, scores) if score >= threshold
        ]

        if relevant_sentences:
            compressed_documents.append(
                {
                    "id": document["id"],
                    "original_text": document["text"],
                    "compressed_text": ". ".join(relevant_sentences) + ".",
                }
            )

    return compressed_documents


# 5. Execute Pipeline
query = "How many annual leave days do employees receive?"

retrieved_documents = retrieve(query, top_k=4)

compressed_documents = compress_context(query, retrieved_documents, threshold=0.30)

# 6. Display Results
print("INITIAL RETRIEVAL")

for document in retrieved_documents:
    print(f"\n{document['id']}:")
    print(document["text"])

print("\n" + "=" * 60)
print("COMPRESSED CONTEXT")

for document in compressed_documents:
    print(f"\n{document['id']}:")
    print(document["compressed_text"])

e:\AI-Engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1176.16it/s]


INITIAL RETRIEVAL

HR-001:

        Employees receive 26 annual leave days per year.
        Leave requests must be approved by the manager.
        Annual leave cannot be transferred without HR approval.
        

HR-004:

        Employees can work remotely two days per week.
        Remote work requires manager approval.
        Employees must attend important office meetings.
        

HR-003:

        Monthly salaries are paid at the end of each month.
        Salary adjustments are reviewed by the finance department.
        Employees can contact payroll for salary questions.
        

HR-002:

        Employees have supplementary health insurance.
        The insurance provider is selected by the HR department.
        Insurance coverage includes dental services.
        

COMPRESSED CONTEXT

HR-001:
Employees receive 26 annual leave days per year. Leave requests must be approved by the manager. Annual leave cannot be transferred without HR approval.

HR-004:
Employees can work 